# Known-person calibration

**Purpose:** Compare population prediction with fixed personal calibration and personal reference procedures.

**Required inputs:** 04 results ZIP; E1_3_Calibration_Assignments_For_Approval.csv.

**Outputs:** Matched personal-trial predictions and calibration metrics.

**Usage:** Run after 04 using the fixed calibration assignment file. Files are read from the notebook’s current working directory unless specified in its configuration.


In [ ]:
# 1 — Dependencies, source contracts, and recording environment
from pathlib import Path
from datetime import datetime, timezone
from io import BytesIO
import hashlib
import json
import platform
import shutil
import sys
import time
import zipfile

import numpy as np
import pandas as pd
import scipy
import sklearn
from sklearn.linear_model import Ridge

PREVIOUS_ZIP = Path('04_Bacek_Foundational_Predictive_Models_results.zip')
ASSIGNMENTS_FILE = Path('E1_3_Calibration_Assignments_For_Approval.csv')
OUTPUT_DIR = Path('11_Bacek_Known_Person_Calibration_results')
OUTPUT_ZIP = Path('11_Bacek_Known_Person_Calibration_results.zip')
NOTEBOOK_FILE = Path('11_Bacek_Known_Person_Calibration.ipynb')

EXPECTED_SOURCE_SHA256 = 'c73ce561e94de52f1dc86128829c43ef536c4cf6b5f4aa9af1ea060409221fa3'
EXPECTED_ASSIGNMENTS_SHA256 = '216a52475c6e7967bfee25965bdfa03237ac5e54d468dd0d134ad22b76b07031'
ALPHAS_PERSON = (0.1, 1.0, 10.0)
K_VALUES = (3, 6)
EVALUATION_PEOPLE = 20
EXPECTED_EVAL_TRIALS = 58
OUTER_FOLDS = {
    1: ('S08', 'S10', 'S16', 'S21'),
    2: ('S11', 'S13', 'S17', 'S20'),
    3: ('S01', 'S02', 'S05', 'S18'),
    4: ('S03', 'S07', 'S12', 'S14', 'S19'),
    5: ('S04', 'S06', 'S09', 'S15'),
}
PROCEDURES = tuple(f'{name}_K{k}' for k in K_VALUES
                   for name in ('A', 'B0_PERSON', 'B1_PERSON', 'C_CAL'))
started_utc = datetime.now(timezone.utc).isoformat()
start_clock = time.perf_counter()

def require(condition, message):
    if not bool(condition):
        raise AssertionError(message)

def sha256_file(path):
    hasher = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(4 * 1024 * 1024), b''):
            hasher.update(chunk)
    return hasher.hexdigest()

require(PREVIOUS_ZIP.is_file(), f'Missing {PREVIOUS_ZIP.name}')
require(ASSIGNMENTS_FILE.is_file(), f'Missing {ASSIGNMENTS_FILE.name}')
require(sha256_file(ASSIGNMENTS_FILE) == EXPECTED_ASSIGNMENTS_SHA256,
        'Assignment manifest fingerprint is incorrect; do not change assignments.')
print('Python:', sys.version.split()[0], '| NumPy:', np.__version__, '| scikit-learn:', sklearn.__version__)
print('Notebook 04 archive:', sha256_file(PREVIOUS_ZIP))
print('E1.3 manifest fingerprint: PASS')

In [ ]:
# 2 — Read and verify Notebook 04 reference predictions and eligible trial ledger
with zipfile.ZipFile(PREVIOUS_ZIP) as z:
    required_members = {
        'run_summary.json', 'eligible_trial_ledger.csv', 'held_out_predictions.csv',
        'selected_models.csv', 'outer_fold_manifest.csv', 'inner_fold_manifest.csv',
    }
    require(required_members.issubset(z.namelist()), 'Notebook 04 archive is incomplete')
    previous = json.loads(z.read('run_summary.json'))
    ledger = pd.read_csv(BytesIO(z.read('eligible_trial_ledger.csv')))
    all_predictions = pd.read_csv(BytesIO(z.read('held_out_predictions.csv')))
    selections = pd.read_csv(BytesIO(z.read('selected_models.csv')))
    outer_manifest = pd.read_csv(BytesIO(z.read('outer_fold_manifest.csv')))
    inner_manifest = pd.read_csv(BytesIO(z.read('inner_fold_manifest.csv')))

require(previous.get('status') == 'PASS', 'Notebook 04 status must be PASS')
require(previous.get('source_sha256') == EXPECTED_SOURCE_SHA256,
        'Notebook 04 source fingerprint differs from authoritative source')
require(len(ledger) == 284 and ledger.trial_uid.is_unique, 'Invalid primary trial ledger')
require(ledger.subject_id.nunique() == 21, 'Expected 21 primary participants')
require(ledger.retained_cycles.sum() == 60757, 'Incorrect primary retained-cycle total')
require(ledger.observed_W_per_kg.notna().all() and
        np.isfinite(ledger.observed_W_per_kg.to_numpy(dtype=np.float64)).all() and
        (ledger.observed_W_per_kg > 0).all(), 'Invalid stored metabolic targets')
require(len(inner_manifest) == 20, 'Expected 20 Notebook 04 inner-fold records')
for fold, group in OUTER_FOLDS.items():
    test = ledger.loc[ledger.outer_fold == fold]
    require(set(test.subject_id) == set(group), f'Unexpected outer group in fold {fold}')
require(len(outer_manifest) == 5, 'Expected five outer-fold manifest rows')

ref = all_predictions.loc[all_predictions.procedure == 'D-ALL9_FIXED'].copy()
require(len(ref) == 284 and ref.trial_uid.is_unique,
        'Expected one fixed-D population prediction per eligible primary trial')
require(np.isfinite(ref.predicted_W_per_kg).all(), 'Population reference has nonfinite predictions')
reference_selection = selections.loc[selections.procedure == 'D-ALL9_FIXED'].copy()
require(len(reference_selection) == 5 and reference_selection.outer_fold.is_unique,
        'Expected five selected fixed-D population models')
joined = ledger.merge(ref[['trial_uid', 'outer_fold', 'subject_id', 'observed_W_per_kg', 'predicted_W_per_kg']],
                      on='trial_uid', how='left', validate='one_to_one',
                      suffixes=('_ledger', '_reference'))
require(joined.predicted_W_per_kg.notna().all(), 'A reference predictions missing trial IDs')
require((joined.outer_fold_ledger == joined.outer_fold_reference).all(),
        'Population reference and trial ledger disagree on outer fold')
require((joined.subject_id_ledger == joined.subject_id_reference).all(),
        'Population reference and trial ledger disagree on participant identity')
require(np.array_equal(joined.observed_W_per_kg_ledger.to_numpy(dtype=np.float64),
                       joined.observed_W_per_kg_reference.to_numpy(dtype=np.float64)),
        'Population reference and ledger targets differ')
trial = ledger.set_index('trial_uid').copy()
trial['population_A_prediction'] = ref.set_index('trial_uid').loc[trial.index, 'predicted_W_per_kg']
require(np.isfinite(trial.population_A_prediction).all(), 'Missing A predictions')
print('Notebook 04 D-ALL9 population reference verified:', len(trial), 'trials')

In [ ]:
# 3 — Verify exact fixed E1.3 personal assignment and information boundaries
assignments = pd.read_csv(ASSIGNMENTS_FILE, dtype={'subject_id': str, 'trial_uid': str, 'role': str})
require(assignments.columns.tolist() == ['subject_id', 'outer_fold', 'trial_uid', 'role'],
        'Unexpected E1.3 assignment columns')
require(len(assignments) == 178 and assignments.trial_uid.is_unique,
        'Expected 178 unique E1.3 assignment records')
require(assignments.role.value_counts().to_dict() == {
    'evaluation': 58, 'calibration_K3': 60, 'calibration_K6_additional': 60,
}, 'Unexpected E1.3 trial roles')
require(assignments.subject_id.nunique() == 20 and 'S12' not in set(assignments.subject_id),
        'E1.3 cohort must contain 20 people and exclude only S12')
require(set(assignments.trial_uid).issubset(set(trial.index)),
        'E1.3 assignment references a trial outside the primary population')
lookup = trial.loc[assignments.trial_uid]
require(np.array_equal(assignments.subject_id.to_numpy(), lookup.subject_id.to_numpy()),
        'Assignment subject IDs do not match source ledger')
require(np.array_equal(assignments.outer_fold.to_numpy(dtype=int),
                       lookup.outer_fold.to_numpy(dtype=int)),
        'Assignment outer-fold IDs do not match source ledger')

personal_assignments = {}
for subject, group in assignments.groupby('subject_id', sort=True):
    role_uids = {name: group.loc[group.role == name, 'trial_uid'].tolist()
                 for name in ('evaluation', 'calibration_K3', 'calibration_K6_additional')}
    evaluation = role_uids['evaluation']
    base = role_uids['calibration_K3']
    additional = role_uids['calibration_K6_additional']
    require(len(base) == len(additional) == 3, f'Unexpected personal budget in {subject}')
    expected_speeds = set(trial.loc[trial.subject_id == subject, 'speed_m_s'])
    eval_speeds = list(trial.loc[evaluation, 'speed_m_s'])
    require(len(evaluation) == len(expected_speeds) and len(set(eval_speeds)) == len(eval_speeds)
            and set(eval_speeds) == expected_speeds,
            f'{subject} must reserve one evaluation trial per represented speed')
    require(set(evaluation).isdisjoint(base + additional) and set(base).isdisjoint(additional),
            f'Personal evaluation and calibration overlap for {subject}')
    personal_assignments[subject] = {
        'outer_fold': int(group.outer_fold.iloc[0]),
        'evaluation': evaluation,
        3: base,
        6: base + additional,
    }
require(sum(len(g['evaluation']) for g in personal_assignments.values()) == 58,
        'Unexpected 58-trial evaluation cohort')
require(sorted(subject for subject, d in personal_assignments.items()
               if len(d['evaluation']) == 2) == ['S08', 'S11'],
        'Two-speed evaluation cohort differs from specification')
print('Personal assignment verification: 20 people, 58 evaluation trials, nested K=3/K=6')

In [ ]:
# 4 — Personal Ridge feature function and small synthetic correctness tests
# Numeric personal-condition features; do not fit a personal scaler.
def personal_condition_features(uids):
    data = trial.loc[list(uids)]
    values = np.column_stack([
        data.speed_m_s.to_numpy(dtype=np.float64) / 1.0,
        (data.relative_cadence.to_numpy(dtype=np.float64) - 1.0) / 0.10,
    ])
    require(values.shape == (len(uids), 2) and np.isfinite(values).all(),
            'Nonfinite or malformed personal condition features')
    return values

def fit_personal_ridge(calibration_uids, alpha):
    features = personal_condition_features(calibration_uids)
    targets = trial.loc[list(calibration_uids), 'observed_W_per_kg'].to_numpy(dtype=np.float64)
    model = Ridge(alpha=float(alpha), fit_intercept=True)
    model.fit(features, targets)
    return model

def rmse(truth, pred):
    target = np.asarray(truth, dtype=np.float64)
    estimate = np.asarray(pred, dtype=np.float64)
    require(len(target) > 0 and target.shape == estimate.shape, 'RMSE inputs misaligned')
    return float(np.sqrt(np.mean((estimate - target)**2, dtype=np.float64)))

def metrics(frame):
    y = frame.observed_W_per_kg.to_numpy(dtype=np.float64)
    p = frame.predicted_W_per_kg.to_numpy(dtype=np.float64)
    e = p - y
    sst = float(np.sum((y - np.mean(y))**2, dtype=np.float64))
    return {
        'n_trials': len(frame), 'n_participants': int(frame.subject_id.nunique()),
        'rmse_W_per_kg': rmse(y, p),
        'mae_W_per_kg': float(np.mean(np.abs(e), dtype=np.float64)),
        'bias_W_per_kg': float(np.mean(e, dtype=np.float64)),
        'r2': float(1.0 - np.sum(e**2, dtype=np.float64) / sst) if sst > 0 else np.nan,
    }

# Check numerical input mapping with a known example (not a Baček target).
test_feature = np.array([[1.1, 2.0]])
test_row = np.array([1.1 / 1.0, (1.20 - 1.0) / 0.10])
require(np.allclose(test_feature[0], test_row, rtol=0, atol=1e-12),
        'Personal two-feature mapping is inconsistent')
print('Personal condition feature mapping and metrics functions available')

In [ ]:
# 5 — Development-only B1-person hyperparameter selection
# For each outer fold, tune each budget using only development participants'
# designated personal calibration/evaluation assignments from the fixed manifest.
meta_scores = []
meta_predictions = []
selected_alphas = []

for fold, held_out_people in OUTER_FOLDS.items():
    development = sorted(set(personal_assignments) - set(held_out_people))
    require(len(development) >= 15, f'Insufficient E1.3-eligible development people: fold {fold}')
    for k in K_VALUES:
        candidate_results = []
        for alpha in ALPHAS_PERSON:
            pooled_true = []
            pooled_pred = []
            for subject in development:
                assignment = personal_assignments[subject]
                require(assignment['outer_fold'] != fold,
                        'Target-fold individual entered personal alpha development set')
                cal_uids = assignment[k]
                eval_uids = assignment['evaluation']
                model = fit_personal_ridge(cal_uids, alpha)
                y_true = trial.loc[eval_uids, 'observed_W_per_kg'].to_numpy(dtype=np.float64)
                y_pred = np.asarray(model.predict(personal_condition_features(eval_uids)), dtype=np.float64)
                pooled_true.extend(y_true)
                pooled_pred.extend(y_pred)
                for uid, truth, pred in zip(eval_uids, y_true, y_pred):
                    meta_predictions.append({
                        'target_outer_fold': fold, 'K': k, 'alpha': float(alpha),
                        'development_subject_id': subject, 'trial_uid': uid,
                        'observed_W_per_kg': float(truth), 'predicted_W_per_kg': float(pred),
                    })
            score = rmse(pooled_true, pooled_pred)
            record = {
                'target_outer_fold': fold, 'K': k, 'alpha': float(alpha),
                'n_development_participants': len(development),
                'n_development_evaluation_trials': len(pooled_true),
                'pooled_personal_evaluation_RMSE_W_per_kg': score,
            }
            meta_scores.append(record)
            candidate_results.append(record)
        winner = min(candidate_results,
                     key=lambda rec: (rec['pooled_personal_evaluation_RMSE_W_per_kg'], rec['alpha']))
        selected_alphas.append({
            'outer_fold': fold, 'K': k, 'selected_alpha': winner['alpha'],
            'development_pooled_RMSE_W_per_kg': winner['pooled_personal_evaluation_RMSE_W_per_kg'],
            'n_development_participants': winner['n_development_participants'],
            'n_development_evaluation_trials': winner['n_development_evaluation_trials'],
            'development_participant_ids': ';'.join(development),
        })

meta_scores_df = pd.DataFrame(meta_scores)
meta_predictions_df = pd.DataFrame(meta_predictions)
selected_alphas_df = pd.DataFrame(selected_alphas)
require(len(meta_scores_df) == 5 * 2 * 3 and len(selected_alphas_df) == 5 * 2,
        'Incorrect development-only personal alpha search coverage')
print(selected_alphas_df[['outer_fold', 'K', 'selected_alpha',
                          'development_pooled_RMSE_W_per_kg']].to_string(index=False))

In [ ]:
# 6 — Produce matched held-out personal predictions at K=3 and K=6
# Every evaluated participant's A reference was originally fitted without
# any of that participant's outcomes (Notebook 04 outer-fold model).
selected_alpha_map = {(int(r.outer_fold), int(r.K)): float(r.selected_alpha)
                      for r in selected_alphas_df.itertuples()}
prediction_rows = []
calibration_evidence = []

for subject, assignment in sorted(personal_assignments.items()):
    fold = assignment['outer_fold']
    eval_uids = assignment['evaluation']
    y_eval = trial.loc[eval_uids, 'observed_W_per_kg'].to_numpy(dtype=np.float64)
    A_eval = trial.loc[eval_uids, 'population_A_prediction'].to_numpy(dtype=np.float64)
    for k in K_VALUES:
        cal_uids = assignment[k]
        cal_data = trial.loc[cal_uids]
        cal_y = cal_data.observed_W_per_kg.to_numpy(dtype=np.float64)
        cal_A = cal_data.population_A_prediction.to_numpy(dtype=np.float64)
        personal_mean = float(np.mean(cal_y, dtype=np.float64))
        mean_residual = float(np.mean(cal_y - cal_A, dtype=np.float64))
        selected_alpha = selected_alpha_map[(fold, k)]
        B1 = fit_personal_ridge(cal_uids, selected_alpha)
        B1_eval = np.asarray(B1.predict(personal_condition_features(eval_uids)), dtype=np.float64)
        values = {
            'A': A_eval,
            'B0_PERSON': np.full(len(eval_uids), personal_mean, dtype=np.float64),
            'B1_PERSON': B1_eval,
            'C_CAL': A_eval + mean_residual,
        }
        calibration_evidence.append({
            'subject_id': subject, 'outer_fold': fold, 'K': k,
            'n_calibration_trials': len(cal_uids), 'n_evaluation_trials': len(eval_uids),
            'calibration_trial_uids': ';'.join(cal_uids),
            'evaluation_trial_uids': ';'.join(eval_uids),
            'B1_selected_alpha': selected_alpha,
            'B0_person_mean_calibration_W_per_kg': personal_mean,
            'C_cal_mean_population_residual_W_per_kg': mean_residual,
        })
        for name, values_array in values.items():
            for uid, obs, pred in zip(eval_uids, y_eval, values_array):
                prediction_rows.append({
                    'procedure': f'{name}_K{k}', 'K': k, 'outer_fold': fold,
                    'subject_id': subject, 'trial_uid': uid,
                    'observed_W_per_kg': float(obs), 'predicted_W_per_kg': float(pred),
                })

predictions_df = pd.DataFrame(prediction_rows)
calibration_df = pd.DataFrame(calibration_evidence)
require(len(predictions_df) == 2 * 4 * 58, 'Expected 464 personal prediction records')
require(len(calibration_df) == 2 * 20, 'Expected 40 subject-budget calibration records')
print('Generated', len(predictions_df), 'held-out prediction rows over',
      predictions_df.subject_id.nunique(), 'people')

In [ ]:
# 7 — Verify calibration boundaries, prediction coverage, and model identity
checks = []
def verify(name, condition):
    passed = bool(condition)
    checks.append({'check': name, 'status': 'PASS' if passed else 'FAIL'})
    require(passed, name)

verify('source_fingerprint_matches', previous['source_sha256'] == EXPECTED_SOURCE_SHA256)
verify('manifest_sha256_matches', sha256_file(ASSIGNMENTS_FILE) == EXPECTED_ASSIGNMENTS_SHA256)
verify('fixed_assignment_roles_and_counts', len(assignments) == 178 and len(personal_assignments) == 20)
verify('exact_58_personal_evaluation_trials', sum(len(a['evaluation']) for a in personal_assignments.values()) == 58)
verify('exactly_20_people_excluding_S12', predictions_df.subject_id.nunique() == 20 and
       'S12' not in set(predictions_df.subject_id))
verify('all_8_procedures_and_464_predictions', len(predictions_df) == 464 and
       set(predictions_df.procedure) == set(PROCEDURES))
verify('all_predictions_finite', np.isfinite(predictions_df.predicted_W_per_kg).all())
verify('unique_procedure_trial_keys', not predictions_df.duplicated(['procedure', 'trial_uid']).any())
verify('all_procedures_same_evaluation_UIDs', all(
       set(subset.trial_uid) == set(assignments.loc[assignments.role == 'evaluation', 'trial_uid'])
       for _, subset in predictions_df.groupby('procedure')))
verify('all_procedures_same_targets', all(
       np.array_equal(subset.set_index('trial_uid').sort_index().observed_W_per_kg.to_numpy(),
                      predictions_df.loc[predictions_df.procedure == 'A_K3']
                      .set_index('trial_uid').sort_index().observed_W_per_kg.to_numpy())
       for _, subset in predictions_df.groupby('procedure')))
verify('A_independent_of_calibration_budget', np.array_equal(
    predictions_df.loc[predictions_df.procedure == 'A_K3']
    .sort_values('trial_uid').predicted_W_per_kg.to_numpy(),
    predictions_df.loc[predictions_df.procedure == 'A_K6']
    .sort_values('trial_uid').predicted_W_per_kg.to_numpy()))
verify('A_matches_Notebook04_D_ALL9', all(np.array_equal(
    subset.sort_values('trial_uid').predicted_W_per_kg.to_numpy(),
    ref.loc[ref.trial_uid.isin(subset.trial_uid)].sort_values('trial_uid')
    .predicted_W_per_kg.to_numpy())
    for name, subset in predictions_df.groupby('procedure') if name.startswith('A_K')))
verify('B1_outer_development_selection_only', all(
    set(str(r.development_participant_ids).split(';')).isdisjoint(OUTER_FOLDS[int(r.outer_fold)])
    for r in selected_alphas_df.itertuples()))
verify('B1_alpha_selections_minimise_development_RMSE', all(
    float(r.selected_alpha) == min(
        meta_scores_df.loc[(meta_scores_df.target_outer_fold == r.outer_fold) &
                           (meta_scores_df.K == r.K)].to_dict('records'),
        key=lambda s: (s['pooled_personal_evaluation_RMSE_W_per_kg'], s['alpha']))['alpha']
    for r in selected_alphas_df.itertuples()))
verify('B1_alpha_grid_exact', set(meta_scores_df.alpha) == set(ALPHAS_PERSON))
verify('nested_calibration_budgets', all(set(g[3]).issubset(set(g[6])) and
       len(g[3]) == 3 and len(g[6]) == 6 for g in personal_assignments.values()))
verify('evaluation_disjoint_from_personal_calibration', all(
       set(g['evaluation']).isdisjoint(g[6]) for g in personal_assignments.values()))
verify('personal_reference_uses_only_K_labels', all(
       len(g[k]) == k for g in personal_assignments.values() for k in K_VALUES))
verify('calibration_residual_equals_saved_evidence', all(
       np.isclose(r.C_cal_mean_population_residual_W_per_kg,
       np.mean(trial.loc[str(r.calibration_trial_uids).split(';'), 'observed_W_per_kg'].to_numpy(dtype=float) -
               trial.loc[str(r.calibration_trial_uids).split(';'), 'population_A_prediction'].to_numpy(dtype=float)),
       rtol=0, atol=1e-13) for r in calibration_df.itertuples()))
verify('predictions_have_correct_outer_folds', all(
       set(group.subject_id).issubset(set(OUTER_FOLDS[int(fold)]))
       for fold, group in predictions_df.groupby('outer_fold')))

checks_df = pd.DataFrame(checks)
print('Verified:', len(checks_df), 'of', len(checks_df), 'checks PASS')

In [ ]:
# 8 — Compute trial-level, outer-fold, participant-level, and paired results
verify('B1_meta_scores_recomputed_from_saved_predictions', all(
    np.isclose(r.pooled_personal_evaluation_RMSE_W_per_kg,
               rmse(group.observed_W_per_kg.to_numpy(), group.predicted_W_per_kg.to_numpy()),
               rtol=0, atol=1e-12)
    for r in meta_scores_df.itertuples()
    for group in [meta_predictions_df.loc[
        (meta_predictions_df.target_outer_fold == r.target_outer_fold) &
        (meta_predictions_df.K == r.K) & (meta_predictions_df.alpha == r.alpha)]]))
verify('B1_meta_predictions_exclude_target_outer_people', all(
    str(r.development_subject_id) not in OUTER_FOLDS[int(r.target_outer_fold)]
    for r in meta_predictions_df.itertuples()))
verify('B0_person_predictions_equal_personal_calibration_means', all(
    np.allclose(predictions_df.loc[
        (predictions_df.subject_id == r.subject_id) &
        (predictions_df.procedure == f'B0_PERSON_K{r.K}'),
        'predicted_W_per_kg'].to_numpy(),
        r.B0_person_mean_calibration_W_per_kg, rtol=0, atol=1e-12)
    for r in calibration_df.itertuples()))
verify('C_cal_predictions_are_A_plus_personal_residual', all(
    np.allclose(
        predictions_df.loc[(predictions_df.subject_id == r.subject_id) &
            (predictions_df.procedure == f'C_CAL_K{r.K}')].sort_values('trial_uid')
            .predicted_W_per_kg.to_numpy(),
        predictions_df.loc[(predictions_df.subject_id == r.subject_id) &
            (predictions_df.procedure == f'A_K{r.K}')].sort_values('trial_uid')
            .predicted_W_per_kg.to_numpy() + r.C_cal_mean_population_residual_W_per_kg,
        rtol=0, atol=1e-12)
    for r in calibration_df.itertuples()))
pooled_rows = []
fold_rows = []
participant_rows = []
for name, part in predictions_df.groupby('procedure', sort=True):
    pooled_rows.append({'procedure': name, **metrics(part)})
    for fold, g in part.groupby('outer_fold'):
        fold_rows.append({'procedure': name, 'outer_fold': int(fold), **metrics(g)})
    for subject, g in part.groupby('subject_id'):
        participant_rows.append({'procedure': name, 'subject_id': subject,
                                 'outer_fold': int(g.outer_fold.iloc[0]), **metrics(g)})
pooled_df = pd.DataFrame(pooled_rows)
fold_df = pd.DataFrame(fold_rows)
participant_df = pd.DataFrame(participant_rows)

comparison_specs = [
    ('PRIMARY_A_vs_C_CAL_K3', 'A_K3', 'C_CAL_K3', 'primary'),
    ('SECONDARY_A_vs_C_CAL_K6', 'A_K6', 'C_CAL_K6', 'secondary'),
    ('SECONDARY_A_vs_B0_PERSON_K3', 'A_K3', 'B0_PERSON_K3', 'secondary'),
    ('SECONDARY_A_vs_B1_PERSON_K3', 'A_K3', 'B1_PERSON_K3', 'secondary'),
    ('SECONDARY_A_vs_B0_PERSON_K6', 'A_K6', 'B0_PERSON_K6', 'secondary'),
    ('SECONDARY_A_vs_B1_PERSON_K6', 'A_K6', 'B1_PERSON_K6', 'secondary'),
    ('DESCRIPTIVE_C_CAL_K3_vs_K6', 'C_CAL_K3', 'C_CAL_K6', 'descriptive'),
]
comparison_rows = []
for comparison_id, a, b, status in comparison_specs:
    a_df = predictions_df.loc[predictions_df.procedure == a].sort_values('trial_uid')
    b_df = predictions_df.loc[predictions_df.procedure == b].sort_values('trial_uid')
    verify('paired_UID_and_target_' + comparison_id,
           np.array_equal(a_df.trial_uid.to_numpy(), b_df.trial_uid.to_numpy()) and
           np.array_equal(a_df.observed_W_per_kg.to_numpy(), b_df.observed_W_per_kg.to_numpy()))
    comparison_rows.append({
        'comparison_id': comparison_id, 'interpretation_status': status,
        'reference': a, 'candidate': b, 'n_evaluation_trials': len(a_df),
        'n_participants': a_df.subject_id.nunique(),
        'reference_RMSE_W_per_kg': metrics(a_df)['rmse_W_per_kg'],
        'candidate_RMSE_W_per_kg': metrics(b_df)['rmse_W_per_kg'],
        'RMSE_improvement_W_per_kg': metrics(a_df)['rmse_W_per_kg'] - metrics(b_df)['rmse_W_per_kg'],
    })
comparisons_df = pd.DataFrame(comparison_rows)
print('All procedures:')
print(pooled_df[['procedure','rmse_W_per_kg','mae_W_per_kg','bias_W_per_kg']].to_string(index=False))
print('Primary E1.3 A minus calibrated A (K=3):',
      f"{comparisons_df.loc[0, 'RMSE_improvement_W_per_kg']:+.6f} W/kg")

In [ ]:
# 9 — Archive complete traceable evidence and verification summary
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
shutil.copyfile(ASSIGNMENTS_FILE, OUTPUT_DIR / ASSIGNMENTS_FILE.name)

def save_csv(filename, frame):
    frame.to_csv(OUTPUT_DIR / filename, index=False, float_format='%.17g')

save_csv('trial_assignments_verified.csv', assignments)
save_csv('personal_calibration_evidence.csv', calibration_df)
save_csv('B1_development_candidate_scores.csv', meta_scores_df)
save_csv('B1_development_candidate_predictions.csv', meta_predictions_df)
save_csv('B1_selected_hyperparameters.csv', selected_alphas_df)
save_csv('held_out_predictions.csv', predictions_df)
save_csv('pooled_metrics.csv', pooled_df)
save_csv('outer_fold_metrics.csv', fold_df)
save_csv('participant_metrics.csv', participant_df)
save_csv('matched_comparisons.csv', comparisons_df)
checks_df = pd.DataFrame(checks)  # Include paired-comparison checks from Section 8.
save_csv('verification_checks.csv', checks_df)
save_csv('population_reference_selected_models.csv', reference_selection)
save_csv('outer_fold_manifest.csv', outer_manifest)

code_sha = None
if NOTEBOOK_FILE.is_file():
    notebook_json = json.loads(NOTEBOOK_FILE.read_text(encoding='utf-8'))
    code_cells = ''.join(''.join(cell.get('source', '')) for cell in notebook_json['cells']
                         if cell['cell_type'] == 'code')
    code_sha = hashlib.sha256(code_cells.encode('utf-8')).hexdigest()

finished_utc = datetime.now(timezone.utc).isoformat()
report = {
    'status': 'PASS', 'experiment_id': 'E1.3',
    'prediction_setting': 'known-person, fixed within-person calibration and held-out condition evaluation',
    'evaluation_status': 'internally held-out, exploratory-informed; not independent-session validation',
    'primary_comparison': 'A_K3 versus C_CAL_K3, improvement is RMSE(A)-RMSE(C_CAL)',
    'primary_difference_W_per_kg': float(comparisons_df.iloc[0].RMSE_improvement_W_per_kg),
    'population': {'participants': 20, 'evaluation_trials': 58, 'S12_excluded_E1_3_only': True,
                   'calibration_budgets': list(K_VALUES), 'total_held_out_prediction_rows': len(predictions_df)},
    'population_A': 'Notebook04 D-ALL9_FIXED out-of-person Ridge predictions; no refitting',
    'personal_B0': 'mean of exactly K designated personal calibration targets',
    'personal_B1': {'features': ['speed_m_s/1.0', '(relative_cadence-1.0)/0.10'],
                    'fit_intercept': True, 'personal_scaler': False,
                    'alpha_candidates': list(ALPHAS_PERSON),
                    'selection': 'per outer fold and K, pooled personal evaluation RMSE on eligible outer-development people; smaller alpha on exact tie'},
    'calibrated_C': 'unchanged A + mean of (personal calibration target - A prediction on same calibration trial)',
    'target_access': 'evaluated person: K calibration labels only; held-out personal evaluation targets used only for final scoring',
    'source_sha256': EXPECTED_SOURCE_SHA256,
    'source_fingerprint_provenance': 'Verified Notebook04 archive, not a new raw-HDF5 read',
    'reference_notebook04_zip_sha256': sha256_file(PREVIOUS_ZIP),
    'reference_notebook04_code_sha256': previous.get('notebook_code_sha256'),
    'assignment_file': ASSIGNMENTS_FILE.name,
    'assignment_sha256': sha256_file(ASSIGNMENTS_FILE),
    'assignment_rows': len(assignments),
    'used_original_personal_assignment_manifest': True,
    'started_utc': started_utc, 'finished_utc': finished_utc,
    'elapsed_seconds': time.perf_counter() - start_clock,
    'software': {'python': sys.version.split()[0], 'numpy': np.__version__,
                 'pandas': pd.__version__, 'scipy': scipy.__version__,
                 'scikit_learn': sklearn.__version__, 'os': platform.platform(),
                 'kernel_python_executable': sys.executable},
    'notebook_filename': NOTEBOOK_FILE.name,
    'notebook_code_sha256': code_sha,
    'verification_checks': len(checks_df),
    'uncertainty': 'deferred to E5.2: 10000 resamples clustered by 20 participants, seed 2026',
}
(OUTPUT_DIR / 'run_summary.json').write_text(json.dumps(report, indent=2, allow_nan=False), encoding='utf-8')
with zipfile.ZipFile(OUTPUT_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for file in sorted(OUTPUT_DIR.iterdir()):
        if file.is_file():
            archive.write(file, arcname=file.name)
print(f'PASS: {len(checks_df)} verification checks')
print('Saved:', OUTPUT_DIR)
print('Saved:', OUTPUT_ZIP)
print('Evidence files:', len(list(OUTPUT_DIR.iterdir())))